# Traffic Signal OpenEnv: Two-Stage Training Pipeline

Stage 1: SFT schema warmup teaches the model to emit valid traffic-action JSON.
Stage 2: GRPO uses the live OpenEnv traffic API only after schema generation passes validation.


In [ ]:
import os
import sys
import subprocess
import torch

print("Installing core dependencies without reinstalling torch...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "numpy", "requests", "matplotlib", "datasets", "wandb",
    "huggingface_hub", "accelerate", "peft", "bitsandbytes",
    "trl", "transformers"
], check=True)

torch_version = torch.__version__.split("+")[0]
cuda_tag = torch.__version__.split("+")[1] if "+" in torch.__version__ else ""
index_args = []
if cuda_tag.startswith("cu"):
    index_args = ["--index-url", f"https://download.pytorch.org/whl/{cuda_tag}"]

print(f"Installing torchvision for torch={torch.__version__}...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir",
    *index_args,
    "torchvision"
], check=True)

print("Installing Unsloth...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)

print("Install complete. Restart kernel once, then rerun from the imports cell.")


In [ ]:
import os
import gc
import json
import re
import time
import random
import requests
import numpy as np
import torch

try:
    import torchvision
    print("torch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
except Exception as exc:
    raise RuntimeError(
        "torchvision import failed. Install a torchvision build matching your torch CUDA build, "
        "restart the kernel, then rerun. Original error: " + repr(exc)
    ) from exc

try:
    import wandb
except Exception:
    wandb = None

from datasets import Dataset
try:
    from unsloth import FastLanguageModel, PatchFastRL
except Exception as exc:
    raise RuntimeError(
        "Unsloth import failed. This is usually a torch/torchvision/CUDA mismatch. "
        "Run the install cell, restart kernel, then rerun imports. Original error: " + repr(exc)
    ) from exc

try:
    from trl import GRPOTrainer, GRPOConfig, SFTTrainer, SFTConfig
except Exception as exc:
    raise RuntimeError(
        "TRL import failed. Run the install cell, restart kernel, then rerun imports. "
        "Original error: " + repr(exc)
    ) from exc

print("Imports ready.")


In [ ]:
PatchFastRL("GRPO", FastLanguageModel)

ENV_URL = os.getenv("ENV_URL", "https://guuru-dev-traffic-signal-openenv-2.hf.space")
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
print("ENV_URL:", ENV_URL)

WANDB_API_KEY = os.getenv("WANDB_API_KEY", "")
use_wandb = bool(wandb) and len(WANDB_API_KEY) >= 40
if not use_wandb:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"
    wandb = None
    print("WandB disabled (missing/invalid WANDB_API_KEY).")
else:
    try:
        wandb.login(key=WANDB_API_KEY, relogin=True)
        wandb.init(project="traffic-signal-rl", name="two-stage-run-1")
        print("WandB initialized.")
    except Exception as e:
        os.environ["WANDB_DISABLED"] = "true"
        os.environ["WANDB_MODE"] = "disabled"
        wandb = None
        use_wandb = False
        print(f"WandB init failed; continuing with WandB disabled: {e}")

r = requests.get(f"{ENV_URL}/health", timeout=30)
assert r.status_code == 200, f"Space not healthy: {r.status_code} {r.text}"
print("Space status:", r.json()["status"])
print("Task count:", r.json()["task_count"])

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Enable GPU before training.")
print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Seeds set to {SEED}.")

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
model_dtype = torch.bfloat16 if use_bf16 else None
print(f"Precision config -> bf16={use_bf16}, fp16={use_fp16}, model_dtype={model_dtype}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=model_dtype,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.generation_config.do_sample = True
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.9
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# Stage 1: supervised schema warmup
VALID_ACTIONS = {"KEEP", "SWITCH", "PHASE_0", "PHASE_1", "PHASE_2", "PHASE_3"}
SCHEMA_PROMPT = (
    'Return exactly one JSON object with this schema: '
    '{"local_actions":{"NW":"ACTION","NE":"ACTION","SW":"ACTION","SE":"ACTION"},"central_action":{}}. '
    'ACTION must be one of KEEP, SWITCH, PHASE_0, PHASE_1, PHASE_2, PHASE_3. '
    'Do not use localAction. Do not use action. No prose.'
)

schema_examples = [
    '{"local_actions":{"NW":"SWITCH","NE":"KEEP","SW":"PHASE_0","SE":"PHASE_1"},"central_action":{}}',
    '{"local_actions":{"NW":"PHASE_2","NE":"SWITCH","SW":"KEEP","SE":"PHASE_0"},"central_action":{}}',
    '{"local_actions":{"NW":"PHASE_1","NE":"PHASE_3","SW":"SWITCH","SE":"KEEP"},"central_action":{}}',
    '{"local_actions":{"NW":"PHASE_0","NE":"PHASE_1","SW":"PHASE_2","SE":"SWITCH"},"central_action":{}}',
    '{"local_actions":{"NW":"SWITCH","NE":"PHASE_0","SW":"PHASE_1","SE":"PHASE_2"},"central_action":{}}',
    '{"local_actions":{"NW":"PHASE_3","NE":"PHASE_2","SW":"SWITCH","SE":"PHASE_0"},"central_action":{}}',
]

rows = []
for _ in range(120):
    for example in schema_examples:
        text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
                {"role": "user", "content": SCHEMA_PROMPT},
                {"role": "assistant", "content": example},
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
        rows.append({"text": text})

sft_dataset = Dataset.from_list(rows)

sft_args = SFTConfig(
    output_dir="./outputs-sft-schema",
    max_steps=80,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    max_seq_length=512,
    packing=False,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    logging_steps=1,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    args=sft_args,
    dataset_text_field="text",
)

sft_trainer.train()
print("Schema SFT warmup complete.")


In [ ]:
# Validate schema generation before starting GRPO
def extract_schema_json(text):
    decoder = json.JSONDecoder()
    best = None
    for idx, char in enumerate(text):
        if char != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[idx:])
        except Exception:
            continue
        if isinstance(obj, dict) and "local_actions" in obj and "central_action" in obj:
            best = obj
    return best

def is_valid_schema(obj):
    if not isinstance(obj, dict):
        return False
    local = obj.get("local_actions")
    if not isinstance(local, dict):
        return False
    if set(local.keys()) != {"NW", "NE", "SW", "SE"}:
        return False
    if any(str(v).upper().strip() not in VALID_ACTIONS for v in local.values()):
        return False
    return obj.get("central_action", {}) == {}

valid_count = 0
for i in range(5):
    messages = [
        {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
        {"role": "user", "content": SCHEMA_PROMPT},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    obj = extract_schema_json(text)
    ok = is_valid_schema(obj)
    valid_count += int(ok)
    print(f"SAMPLE {i + 1} valid={ok} extracted={obj}")

SCHEMA_READY = valid_count >= 3
print(f"Schema validation: {valid_count}/5 valid")
if not SCHEMA_READY:
    raise RuntimeError("Schema warmup insufficient. Do not start GRPO; rerun SFT warmup or increase max_steps.")


In [ ]:
# Stage 2: GRPO environment fine-tuning, gated on schema readiness
if not globals().get("SCHEMA_READY", False):
    raise RuntimeError("SCHEMA_READY is false. Run schema validation before GRPO.")

PROMPT_BANK = [
    'Return exactly one compact JSON object: {"local_actions":{"NW":"SWITCH","NE":"KEEP","SW":"PHASE_0","SE":"PHASE_1"},"central_action":{}}',
    'Choose traffic actions for NW NE SW SE. Use valid JSON only. Include at least one non-KEEP action.',
    'Optimize congestion. Return local_actions with NW NE SW SE and central_action empty. JSON only.',
    'Prevent spillback. Use actions KEEP SWITCH PHASE_0 PHASE_1 PHASE_2 PHASE_3. JSON only.',
    'Emergency-aware control. Return compact JSON action only.',
] * 8

train_dataset = Dataset.from_dict({"prompt": PROMPT_BANK})
collected_data = []
GLOBAL_EPISODE = 0

def safe_post(url, payload, retries=16, timeout=60):
    for attempt in range(retries):
        try:
            rr = requests.post(url, json=payload, timeout=timeout)
            if rr.status_code in (429, 500, 502, 503, 504):
                wait = min(45, 2 * (attempt + 1)) + random.uniform(0.0, 1.0)
                print(f"HTTP {rr.status_code}. Waiting {wait:.1f}s ({attempt+1}/{retries})")
                time.sleep(wait)
                continue
            rr.raise_for_status()
            return rr
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = min(30, 2 + attempt)
            print(f"Network/timeout attempt {attempt+1}/{retries}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} retries: {url}")

def parse_action(completion):
    obj = extract_schema_json(completion)
    if is_valid_schema(obj):
        clean = {k: str(v).upper().strip() for k, v in obj["local_actions"].items()}
        return {"local_actions": clean, "central_action": {}}, False
    return {"local_actions": {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"}, "central_action": {}}, True

def reward_fn(prompts, completions, **kwargs):
    global GLOBAL_EPISODE
    rewards = []
    for _, completion in zip(prompts, completions):
        GLOBAL_EPISODE += 1
        episode = GLOBAL_EPISODE
        task_id = "medium_dynamic" if episode < 20 else "hard_multi"
        action, is_hallucinated = parse_action(completion)

        if is_hallucinated:
            reward = -6.0
            print(f"[DEBUG] ep={episode} reward={reward:.4f} hallucinated=True raw={completion[:160]!r}")
            collected_data.append({"episode_reward": reward, "mean_queue": 0.0, "final_score": 0.0, "throughput": 0.0, "step_count": 0, "is_hallucinated": 1.0})
            rewards.append(reward)
            continue

        if all(v == "KEEP" for v in action["local_actions"].values()):
            reward = -3.0
            print(f"[DEBUG] ep={episode} reward={reward:.4f} all_keep=True action={action}")
            collected_data.append({"episode_reward": reward, "mean_queue": 0.0, "final_score": 0.0, "throughput": 0.0, "step_count": 0, "is_hallucinated": 0.0})
            rewards.append(reward)
            continue

        safe_post(f"{ENV_URL}/reset", {"task_id": task_id, "central_enabled": True})
        episode_reward = 0.0
        done = False
        step_count = 0
        info = {}
        latency_ms = 0.0

        while not done and step_count < 25:
            t0 = time.time()
            try:
                step_res = safe_post(f"{ENV_URL}/step", action)
            except requests.HTTPError as exc:
                if getattr(exc.response, "status_code", None) == 422:
                    episode_reward = -5.0
                    print(f"[DEBUG] ep={episode} reward={episode_reward:.4f} http422=True action={action}")
                    break
                raise
            latency_ms = (time.time() - t0) * 1000
            data = step_res.json()
            info = data.get("info", {})
            episode_reward += float(data.get("reward", 0.0))
            done = data.get("done", False)
            step_count += 1
            time.sleep(0.05)

        print(f"[DEBUG] ep={episode} reward={episode_reward:.4f} task={task_id} valid=True action={action}")
        collected_data.append({
            "episode_reward": episode_reward,
            "mean_queue": info.get("mean_queue", 0.0),
            "final_score": info.get("final_score", 0.0),
            "throughput": info.get("throughput", 0.0),
            "step_count": step_count,
            "is_hallucinated": 0.0,
        })
        rewards.append(episode_reward)
    return rewards

grpo_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    max_steps=80,
    max_prompt_length=512,
    max_completion_length=96,
    num_generations=4,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    logging_steps=1,
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=grpo_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
grpo_trainer.args.report_to = []
grpo_trainer.callback_handler.callbacks = [cb for cb in grpo_trainer.callback_handler.callbacks if "wandb" not in cb.__class__.__name__.lower()]

print("Starting GRPO environment training...")
torch.cuda.empty_cache()
gc.collect()
grpo_trainer.train()

model.save_pretrained("outputs/traffic-lora")
tokenizer.save_pretrained("outputs/traffic-lora")
print("Adapter weights saved to outputs/traffic-lora")


In [ ]:
if use_wandb and wandb:
    wandb.finish()
print("WandB run complete.")


In [ ]:
# Auto-save adapter and plots to HF Hub
import os
import sys
from huggingface_hub import HfApi

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Add it as a Kaggle secret or set os.environ['HF_TOKEN'].")

api = HfApi(token=HF_TOKEN)

assert os.path.exists("outputs/traffic-lora"), "FAIL: outputs/traffic-lora not found. Training may not have completed."
print("Model weights found.")
print("Contents:", os.listdir("outputs/traffic-lora"))

api.upload_folder(
    folder_path="outputs/traffic-lora",
    repo_id="Guuru-DEV/traffic-signal-openenv-2",
    repo_type="space",
    path_in_repo="outputs/traffic-lora",
    token=HF_TOKEN,
)
print("Adapter uploaded to HF Space: https://huggingface.co/spaces/Guuru-DEV/traffic-signal-openenv-2")

os.makedirs("plots", exist_ok=True)
try:
    sys.path.insert(0, ".")
    from env.metrics_exporter import generate_training_plots
    if "collected_data" not in globals():
        raise RuntimeError("collected_data not found. Ensure GRPO cell completed.")
    generate_training_plots(collected_data, "plots")
    print("Plots generated in plots/")
except Exception as e:
    print(f"Plot generation skipped (non-fatal): {e}")

print("Training artifact save step complete.")
